<a href="https://colab.research.google.com/github/kulsumm/AAK/blob/main/UCTP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# constraints_hard.py

import pandas as pd
from ortools.sat.python import cp_model


# ===================================================================
# 1. EACH SECTION MUST BE ASSIGNED EXACTLY ONE (ROOM, TIMESLOT)
# ===================================================================
def add_section_single_assignment(model, x, section_list, room_list, slot_list):
    """
    Hard Constraint:
    Every section must appear exactly ONCE in the schedule.
    One section → one room → one timeslot.
    """
    for sid in section_list:  # loop over all section IDs
        model.Add(sum(
            x[(sid, rid, tid)]       # decision variable = section s in room r at time t
            for rid in room_list # check all rooms
            for tid in slot_list # check all times
            if (sid, rid, tid) in x  # skip invalid combos
        ) == 1)               # enforce exactly 1 assignment


In [ ]:
# ===================================================================
# 2. ROOM CAPACITY ≥ ENROLMENT
# ===================================================================
def add_room_capacity_constraint(model, x, room_capacity, Section_enrol):
    """
    Hard Constraint:
    Prevent assigning a section to a room that is too small.
    """
    for (sid, rid, tid), var in x.items():  # loop through every possible assignment
        if room_capacity[rid] < Section_enrol[sid]:
            model.Add(var == 0)       # forbid assignment

In [1]:
# ===================================================================
# 3. ROOM TYPE MUST MATCH COURSE TYPE
# ===================================================================
def add_room_type_constraint(model, x, Required_Room_Type, Room_Type):
    """
    Hard Constraint:
    Each course requires a certain room type:
    e.g., LAB → Lab room, Lecture → Lecture room, Studio → Studio room.
    """
    # convert dataframe to dictionary: section → required room type
    sec_req_type = sections_df.set_index("Section_ID")["Req_Room_Type"].to_dict()

    for (s, r, t), var in x.items():   # loop all potential assignments
        req_type = sec_req_type.get(s) # required type for this section
        room_type = room_type_clean.get(r) # actual room type
        if req_type != room_type:      # mismatch?
            model.Add(var == 0)        # forbid assignment

In [3]:
# ===================================================================
# 4. CAMPUS / ZONE MATCH
# ===================================================================
def add_campus_match_constraint(model, x, sections_df, room_campus_map):
    """
    Hard Constraint:
    Section campus must match room campus.
    E.g. MAM → male campus, MAW → female campus, etc.
    """
    # section → campus mapping
    sec_campus = sections_df.set_index("Section_ID")["Campus"].to_dict()

    for (s, r, t), var in x.items():
        section_c = sec_campus.get(s)        # required campus
        room_c = room_campus_map.get(r)      # room campus
        if section_c != room_c:              # mismatch?
            model.Add(var == 0)              # forbid assignment

    #ORRRR

def add_room_match_constraint(model, x, Required_Room_Type, Room_Type):
    for (sid, rid, tid), var in x.items():
        if Required_Room_Type[sid] != Room_Type[rid]:
            model.Add(var == 0)
def add_section_room_constraint(model, x, Section_Campus, Room_Campus):
    for (sid, rid, tid), var in x.items():
        if Section_Campus[sid] != Room_Campus[rid]:
            model.Add(var == 0)

In [ ]:
# ===================================================================
# 5. DEPARTMENT ROOM CONSTRAINT
# ===================================================================
def add_department_room_constraint(model, x, sections_df, dept_owned_rooms, room_owner):
    """
    Hard Constraint:
    Department-owned rooms may ONLY be used by that department.
    """
    # map section → its department
    sec_dept = sections_df.set_index("Section_ID")["Department_Code"].to_dict()

    for (s, r, t), var in x.items():
        if r in dept_owned_rooms:       # if this room is department-owned
            sec_d = sec_dept.get(s)     # department of this section
            owner_d = room_owner.get(r) # owning department of room
            if sec_d != owner_d:        # mismatch?
                model.Add(var == 0)     # forbid assignment

    #ORRRRR

def HC_department_room(model, x, Section_Department, Room_Department):
    for (sid, rid, tid), var in x.items():
        if Room_Department.get(rid) is not None:  # room is owned
            if Section_Department[sid] != Room_Department[rid]:
                model.Add(var == 0)


In [ ]:

# ===================================================================
# 6. LAB ROOM CONSTRAINT
# ===================================================================
def add_lab_room_constraint(model, x, course_of_section, lab_allowed_rooms):
    """
    Hard Constraint:
    Lab courses can only use specific lab rooms.
    """
    for (s, r, t), var in x.items():
        cnum = course_of_section.get(s)           # find course number
        allowed = lab_allowed_rooms.get(cnum)     # allowed rooms for this course
        if allowed is not None and r not in allowed:
            model.Add(var == 0)                   # forbid wrong lab room

#ORRRRRR
def HC_lab_rooms(model, x, Course_Number_of_Section, Lab_Allowed_Rooms):
    for (sid, rid, tid), var in x.items():
        cnum = Course_Number_of_Section[sid]
        allowed = Lab_Allowed_Rooms.get(cnum)
        if allowed is not None and rid not in allowed:
            model.Add(var == 0)



In [ ]:
# ===================================================================
# 7. NO TWO SECTIONS IN SAME ROOM AT SAME TIME
# ===================================================================
def add_room_no_overlap(model, x, section_list, room_list, slot_list):
    """
    Hard Constraint:
    A room cannot host two sections at the same time.
    """
    for r in room_list:            # for each room
        for t in slot_list:        # for each timeslot
            model.Add(sum(
                x[(s, r, t)]       # check if section s is in room r at time t
                for s in section_list
                if (s, r, t) in x
            ) <= 1)                # ≤1 ensures no conflicts


#ORRRRRRR
def HC_room_no_overlap(model, x, Section_IDs, Room_IDs, Slot_IDs):
    for rid in Room_IDs:
        for tid in Slot_IDs:
            model.Add(sum(
                x[(sid, rid, tid)]
                for sid in Section_IDs
                if (sid, rid, tid) in x
            ) <= 1)


In [6]:
# ===================================================================
# 8. INSTRUCTOR CANNOT TEACH AT TWO PLACES AT SAME TIME
# ===================================================================
def add_instructor_no_overlap(model, x, section_list, room_list, slot_list,
                              course_of_section, course_instr_map, instructor_set):
    """
    Hard Constraint:
    Instructor cannot teach two sections simultaneously.
    """
    for instr in instructor_set:         # loop all instructors
        for t in slot_list:              # check each timeslot
            vars_same_time = []          # store all assignments involving this instructor at time t

            for s in section_list:
                cnum = course_of_section.get(s)  # course assigned to section
                if cnum is None:
                    continue
                # does this instructor teach this course?
                if instr in course_instr_map.get(cnum, []):
                    for r in room_list:
                        if (s, r, t) in x:
                            vars_same_time.append(x[(s, r, t)])

            if vars_same_time:                   # if instructor has any sections at time t
                model.Add(sum(vars_same_time) <= 1)   # enforce no overlap



#ORRRRRRRR

def HC_instructor_no_overlap(model, x, Instructor_Courses,
                              Course_Number_of_Section, Section_IDs, Room_IDs, Slot_IDs):

    for instr, course_list in Instructor_Courses.items():
        for tid in Slot_IDs:
            vars_same_time = []
            for sid in Section_IDs:
                if Course_Number_of_Section[sid] in course_list:
                    for rid in Room_IDs:
                        if (sid, rid, tid) in x:
                            vars_same_time.append(x[(sid, rid, tid)])
            if vars_same_time:
                model.Add(sum(vars_same_time) <= 1)


In [5]:
# ===================================================================
# 9. STUDENT CANNOT ATTEND TWO CLASSES AT SAME TIME
# ===================================================================
def add_student_no_overlap(model, x, room_list, slot_list,
                           registration_df, sections_df):
    """
    Hard Constraint:
    A student cannot be scheduled in two sections that overlap.
    Uses Registration.csv to map Student_ID → courses → sections.
    """

    # map student → list of their courses
    student_courses = (
        registration_df.groupby("Student_ID")["Course_Number"]
        .apply(list)
        .to_dict()
    )

    # map course → its offered sections
    course_to_sections = (
        sections_df.groupby("Course_Number")["Section_ID"]
        .apply(list)
        .to_dict()
    )

    for student, courses in student_courses.items():
        # get all sections this student might attend
        student_sections = []
        for c in courses:
            student_sections.extend(course_to_sections.get(c, []))

        student_sections = list(set(student_sections)) # remove duplicates

        # no two of these sections can be in the same slot
        for t in slot_list:
            model.Add(sum(
                x[(s, r, t)]                  # if student attends section s at time t in room r
                for s in student_sections
                for r in room_list
                if (s, r, t) in x
            ) <= 1)                           # must be at most 1






#ORRRRRRRRRRRRRRRRRRRRRR


def HC_student_no_overlap(model, x, Student_Courses,
                           Course_Sections, Room_IDs, Slot_IDs):

    for student, course_list in Student_Courses.items():
        # all sections student attends
        student_sections = []
        for c in course_list:
            student_sections.extend(Course_Sections.get(c, []))

        for tid in Slot_IDs:
            model.Add(sum(
                x[(sid, rid, tid)]
                for sid in student_sections
                for rid in Room_IDs
                if (sid, rid, tid) in x
            ) <= 1)


In [ ]:
# ===================================================================
# 10. MULTI-ROW CONSISTENCY CONSTRAINT
# ===================================================================
def add_multirow_consistency(model, x, sections_df, room_list, slot_list):
    """
    Hard Constraint:
    If multiple rows represent the same logical section (e.g., lecture A split over dataset),
    force all of them to share the exact same (room, slot).
    Only used if 'Logical_ID' exists.
    """

    # group all rows by Logical_ID
    if "Logical_ID" not in sections_df.columns:
        return

    logical_groups = sections_df.groupby("Logical_ID")["Section_ID"].unique()

    for logical_id, sec_ids in logical_groups.items():
        sec_ids = list(sec_ids)
        if len(sec_ids) <= 1:
            continue

        base = sec_ids[0]  # reference section
        for other in sec_ids[1:]:
            for r in room_list:
                for t in slot_list:
                    # if both assignments exist
                    if (base, r, t) in x and (other, r, t) in x:
                        # force them to be identical
                        model.Add(x[(base, r, t)] == x[(other, r, t)])

In [ ]:
# ===================================================================
# 11.Prefer evening slots for graduate courses
# ===================================================================
def HC_graduate_evening_only(model, x, Level_of_Section, Evening_Slots):
    """
    Hard Constraint:
    Graduate sections (GR) can ONLY be scheduled in evening slots.
    Undergraduate (UG) sections are unrestricted.
    """
    for (sid, rid, tid), var in x.items():

        # if this is a graduate section
        if Level_of_Section[sid] == "GR":

            # if the slot is NOT in evening allowed slots
            if tid not in Evening_Slots:

                # forbid this assignment
                model.Add(var == 0)
